# Pre-processing

### 1. Importing libraries

In [2]:
import pandas as pd
import numpy as np

In [ ]:
pd.set_option("display.max_columns", None)

### 2. Loading training set

In [3]:
df = pd.read_csv("train.csv")
df.head()

,MSISDN,CRM_BUSINESS_LINE_ORIG_Mobile,CRM_1519_PACK,CRM_1521_PACK,CRM_1522_PACK,CRM_1523_PACK,CRM_1524_PACK,CRM_1525_PACK,CRM_1526_PACK,CRM_1527_PACK,...,BIL_4465_PACK,BIL_4466_PACK,BIL_4467_PACK,BIL_4468_PACK,BIL_4504_PACK,BIL_4506_PACK,BIL_4507_PACK,BIL_4508_PACK,BIL_4509_PACK,BIL_4510_PACK
0,4f1b9e575ac7ea93b4a978b3095d7ee1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,01908281632545a2fc22d4a6365f2b27,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,f6f0c782716d8928928407c4e426be25,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,a543952f2405c440fc40f85e8adfdfd6,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,63e3cdf2eff067ce9af9f9f8530b16eb,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


The training set contains both:

· CRM input variables
· Billing target variables
· The customer identifier

### 3. Separate identifiers, CRM features, and Billing targets

In [4]:
id_col = "MSISDN"
crm_cols = [c for c in df.columns if c.startswith("CRM")]
bil_cols = [c for c in df.columns if c.startswith("BIL")]

assert id_col in df.columns
assert len(crm_cols) == 745
assert len(bil_cols) == 731

This creates three logical groups:

· MSISDN: customer identifier 
· crm_cols: 745 input features 
· bil_cols: 731 output targets

### 4. Convert binary columns to uint8

In [5]:
df[crm_cols] = df[crm_cols].astype(np.uint8)
df[bil_cols] = df[bil_cols].astype(np.uint8)

Since the variables contain only 0 and 1, uint8 is appropriate and uses much less memory than the default integer type.

The same conversion is applied to the test CRM columns.

### 5. Validate the test dataset
The test validation checks that:

· The test data contains the same CRM columns as training.
· Customer IDs are not missing.
· Customer IDs are unique.
· All CRM values are binary.
· There are no missing values.
· This is essential because the model must receive exactly the same feature structure during training and prediction.

In [6]:
test = pd.read_csv("test.csv")
test_crm_cols = [c for c in test.columns if c.startswith("CRM")]

assert list(test.columns) == [id_col] + test_crm_cols
assert test_crm_cols == crm_cols
assert test[id_col].notna().all()
assert not test[id_col].duplicated().any()

# Apply the same binary dtype and validate the test inputs.
test[crm_cols] = test[crm_cols].astype(np.uint8)
test_values = set(pd.unique(test[crm_cols].to_numpy().ravel()))
assert test_values.issubset({0, 1})
assert int(test[crm_cols].isna().sum().sum()) == 0

print(f"Test rows: {len(test):,}")
print(f"Test CRM features: {len(test_crm_cols)}")
print(f"Test active values: {sorted(test_values)}")

Test rows: 97,100
Test CRM features: 745
Test active values: [np.uint8(0), np.uint8(1)]


### 6. Constant-column analysis

Selecting CRM and Billing columns for the final datasets

The two CRM columns `CRM_2396_PACK` and `CRM_2398_PACK` are constant at 0 in the training set, but they contain both 0 and 1 in the test set. They are therefore retained instead of being removed, because excluding them would discard information that may be relevant for test predictions. Constant-column analysis is documented, but no CRM columns are excluded from the final train/test feature set.

In [7]:
# Keep every CRM input so test-only variation is not discarded.
# The Billing targets are still checked for constant columns.
active_crm = crm_cols.copy()
active_bil = [c for c in bil_cols if df[c].min() != df[c].max()]

train_constant_crm = [c for c in crm_cols if df[c].min() == df[c].max()]
test_constant_crm = [c for c in crm_cols if test[c].min() == test[c].max()]

print(f"CRM Features retained: {len(active_crm)} / {len(crm_cols)}")
print(f"Billing Targets retained: {len(active_bil)} / {len(bil_cols)}")
print(f"Constant CRM columns in train, retained: {train_constant_crm}")
print(f"Constant CRM columns in test: {test_constant_crm}")

CRM Features retained: 745 / 745
Billing Targets retained: 731 / 731
Constant CRM columns in train, retained: ['CRM_2396_PACK', 'CRM_2398_PACK']
Constant CRM columns in test: ['CRM_2145_PACK', 'CRM_1866_PACK', 'CRM_2208_PACK', 'CRM_1891_PACK', 'CRM_1904_PACK', 'CRM_2252_PACK', 'CRM_2272_PACK', 'CRM_2279_PACK', 'CRM_2282_PACK', 'CRM_2284_PACK', 'CRM_2292_PACK', 'CRM_2293_PACK', 'CRM_2297_PACK', 'CRM_2315_PACK', 'CRM_2325_PACK', 'CRM_2326_PACK', 'CRM_2344_PACK', 'CRM_2345_PACK', 'CRM_2348_PACK', 'CRM_2355_PACK', 'CRM_2363_PACK', 'CRM_2365_PACK', 'CRM_2366_PACK', 'CRM_2367_PACK', 'CRM_2368_PACK', 'CRM_2370_PACK', 'CRM_2374_PACK', 'CRM_2377_PACK', 'CRM_2385_PACK', 'CRM_2387_PACK', 'CRM_2391_PACK', 'CRM_2392_PACK', 'CRM_2397_PACK']


### 7. Create active matrices

In [ ]:
X_active = df[active_crm]
Y_active = df[active_bil]

Finding clients with equal packages and different billing methods. Then, we proceed to choose the billing mthod most predominant between these clients

### 8. Resolve repeated CRM configurations

In [ ]:
combined_df = pd.concat([X_active, Y_active], axis=1)

This combines the CRM and Billing information so repeated CRM configurations can be analyzed.

In [ ]:
# The grouping uses the complete CRM and Billing vectors:
group_counts = (
    combined_df.groupby(active_crm + active_bil, sort=False)
    .size()
    .reset_index(name="_count")
    .sort_values("_count", ascending=False, kind="stable")
) # This counts how often each complete CRM-Billing combination appears
df_clean = (
    group_counts
    .drop_duplicates(active_crm) # Keeps only the billing configuration for each CRM configuration
    .drop(columns="_count")
    .reset_index(drop=True)
)

X_clean = df_clean[active_crm]
Y_clean = df_clean[active_bil]

Because the combinations were sorted by frequency first, the selected Billing configuration is the most frequent one.

This is a majority-rule noise-cleaning strategy:

- Same CRM configuration
- Several Billing configurations
- Keep the most frequent complete Billing vector

In [ ]:
df_clean.to_csv('cleaned_telecom_data.csv', index=False)

This reduces the training data from:

In [ ]:
df.shape

(187442, 1477)

To:

In [ ]:
df_clean.shape

(67434, 1476)

The reduction is approximately 64%, because only one row is retained for each unique CRM configuration.

### 9. Audit pre-processing

In [ ]:
train_binary_cols = crm_cols + bil_cols
binary_values = set(pd.unique(df[train_binary_cols].to_numpy().ravel()))

train_input_cols = [c for c in df.columns if c.startswith("CRM")]
test_input_cols = [c for c in test.columns if c.startswith("CRM")]

crm_group_sizes = df.groupby(crm_cols, sort=False).size()
conflicting_groups = (
    df.groupby(crm_cols, sort=False)[bil_cols]
    .nunique()
    .gt(1)
    .any()
)

print("Train rows:", len(df))
print("Train columns:", df.shape[1])
print("Test rows:", len(test))
print("Test columns:", test.shape[1])
print("CRM columns train and test match:", train_input_cols == test_input_cols)
print("Missing values:", int(df[train_binary_cols].isna().sum().sum()))
print("Test missing values:", int(test[crm_cols].isna().sum().sum()))
print("Duplicated ids:", int(df[id_col].duplicated().sum()))
print("Test duplicated ids:", int(test[id_col].duplicated().sum()))
print("Duplicates CRM groups:", int((crm_group_sizes > 1).sum()))
print("Conflicting CRM groups:", int(conflicting_groups.sum()))
print("Cleaned rows:", len(df_clean))
print("Cleaned features plus targets:", df_clean.shape[1])

Train rows: 187442
Train columns: 1477
Test rows: 97100
Test columns: 746
CRM columns train and test match: True
Missing values: 0
Test missing values: 0
Duplicated ids: 0
Test duplicated ids: 0
Duplicates CRM groups: 9581
Conflicting CRM groups: 407
Cleaned rows: 67434
Cleaned features plus targets: 1476


### Additional preprocessing analysis

In [ ]:
# Univariate and train/test distribution analysis
crm_prevalence = df[crm_cols].mean().sort_values(ascending=False)
bil_prevalence = df[bil_cols].mean().sort_values(ascending=False)
train_active_counts = df[crm_cols].sum(axis=1)
test_active_counts = test[crm_cols].sum(axis=1)

feature_drift = pd.DataFrame({
    "train_prevalence": df[crm_cols].mean(),
    "test_prevalence": test[crm_cols].mean(),
})
feature_drift["absolute_difference"] = (
    feature_drift["test_prevalence"] - feature_drift["train_prevalence"]
).abs()

print("Top CRM features by training prevalence:")
display(crm_prevalence.head(15).to_frame("prevalence"))
print("Top Billing targets by training prevalence:")
display(bil_prevalence.head(15).to_frame("prevalence"))
print("CRM active-count summary:")
display(pd.DataFrame({"train": train_active_counts.describe(), "test": test_active_counts.describe()}))
print("Largest train/test prevalence differences:")
display(feature_drift.sort_values("absolute_difference", ascending=False).head(15))

Top CRM features by training prevalence:


,prevalence
CRM_BUSINESS_LINE_ORIG_Mobile,0.998725
CRM_SUBSCRIBER_TYPE_ORIG_Prepaid,0.798444
CRM_1537_PACK,0.727409
CRM_SUBSCRIBER_STATUS_ORIG_Active,0.685087
CRM_1542_PACK,0.520918
CRM_1533_PACK,0.520908
CRM_1529_PACK,0.520033
CRM_1541_PACK,0.519009
CRM_1532_PACK,0.514623
CRM_2076_PACK,0.428463


Top Billing targets by training prevalence:


,prevalence
BIL_BUSINESS_LINE_ORIG_Mobile,0.998629
BIL_SUBSCRIBER_TYPE_ORIG_Prepaid,0.798002
BIL_3748_PACK,0.727542
BIL_SUBSCRIBER_STATUS_ORIG_Active,0.687199
BIL_3933_PACK,0.520897
BIL_3934_PACK,0.520844
BIL_3921_PACK,0.520017
BIL_3927_PACK,0.519057
BIL_3922_PACK,0.514426
BIL_3749_PACK,0.428202


CRM active-count summary:


,train,test
count,187442.000000,97100.000000
mean,15.293563,16.350793
std,4.381086,2.417609
min,3.000000,2.000000
25%,14.000000,15.000000
50%,15.000000,16.000000
75%,17.000000,18.000000
max,50.000000,34.000000


Largest train/test prevalence differences:


,train_prevalence,test_prevalence,absolute_difference
CRM_1529_PACK,0.520033,0.763913,0.243881
CRM_1542_PACK,0.520918,0.764562,0.243644
CRM_1533_PACK,0.520908,0.764542,0.243634
CRM_1541_PACK,0.519009,0.760412,0.241403
CRM_1532_PACK,0.514623,0.748198,0.233575
CRM_2076_PACK,0.428463,0.230288,0.198175
CRM_2077_PACK,0.423219,0.225767,0.197452
CRM_2087_PACK,0.417414,0.229691,0.187723
CRM_2088_PACK,0.402098,0.218980,0.183117
CRM_1527_PACK,0.281132,0.443584,0.162452


In [ ]:
# Bivariate CRM-to-Billing screening.
# Pearson correlation is the phi coefficient for pairs of binary variables.
crm_variance = df[crm_cols].var()
bil_variance = df[bil_cols].var()
valid_crm = crm_variance[crm_variance > 0].index
valid_bil = bil_variance[bil_variance > 0].index

crm_centered = df[valid_crm].to_numpy(dtype=np.float32)
bil_centered = df[valid_bil].to_numpy(dtype=np.float32)
crm_centered -= crm_centered.mean(axis=0)
bil_centered -= bil_centered.mean(axis=0)
association = (crm_centered.T @ bil_centered) / (
    np.sqrt((crm_centered ** 2).sum(axis=0))[:, None]
    * np.sqrt((bil_centered ** 2).sum(axis=0))[None, :]
)
association_table = (
    pd.DataFrame(association, index=valid_crm, columns=valid_bil)
    .stack()
    .rename("phi")
    .abs()
    .sort_values(ascending=False)
    .head(25)
    .reset_index()
)
association_table.columns = ["CRM_feature", "Billing_target", "absolute_phi"]
print("Strongest binary CRM/Billing associations:")
display(association_table)

# Multivariate analysis: repeated CRM packages and their Billing consistency.
crm_signatures = df[crm_cols].astype(np.uint8).astype(str).agg("".join, axis=1)
signature_summary = (
    pd.DataFrame({"crm_signature": crm_signatures})
    .groupby("crm_signature")
    .size()
    .rename("rows")
    .to_frame()
)
signature_summary["share_of_rows"] = signature_summary["rows"] / len(df)
print("Repeated CRM package summary:")
display(signature_summary["rows"].describe())
print(f"Unique CRM signatures: {len(signature_summary):,} / {len(df):,} rows")

Strongest binary CRM/Billing associations:


,CRM_feature,Billing_target,absolute_phi
0,CRM_1533_PACK,BIL_3934_PACK,0.995788
1,CRM_1533_PACK,BIL_3933_PACK,0.995706
2,CRM_1542_PACK,BIL_3933_PACK,0.995577
3,CRM_1542_PACK,BIL_3934_PACK,0.995529
4,CRM_1541_PACK,BIL_3927_PACK,0.995523
5,CRM_1529_PACK,BIL_3921_PACK,0.995309
6,CRM_1532_PACK,BIL_3922_PACK,0.995247
7,CRM_1538_PACK,BIL_3935_PACK,0.995181
8,CRM_2088_PACK,BIL_3744_PACK,0.995110
9,CRM_2087_PACK,BIL_3745_PACK,0.995041


Repeated CRM package summary:


count    67434.000000
mean         2.779636
std         40.230352
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max       8114.000000
Name: rows, dtype: float64

Unique CRM signatures: 67,434 / 187,442 rows


In [ ]:
# Export aligned cleaned train and test data.
# The ID is kept in the test output; it is excluded from model features.
cleaned_test = test[[id_col] + active_crm].copy()
cleaned_train = df_clean[active_crm + active_bil].copy()

assert list(cleaned_test.columns) == [id_col] + active_crm
assert list(cleaned_train.columns) == active_crm + active_bil
assert cleaned_test[active_crm].dtypes.eq(np.dtype("uint8")).all()
assert cleaned_train[active_crm + active_bil].isin([0, 1]).all().all()

cleaned_test.to_csv("cleaned_test.csv", index=False)
cleaned_train.to_csv("cleaned_telecom_data.csv", index=False)

print(f"Cleaned train shape: {cleaned_train.shape}")
print(f"Cleaned test shape: {cleaned_test.shape}")
print(f"Removed constant CRM features: {len(crm_cols) - len(active_crm)}")
print(f"Removed constant Billing targets: {len(bil_cols) - len(active_bil)}")

Cleaned train shape: (67434, 1476)
Cleaned test shape: (97100, 746)
Removed constant CRM features: 0
Removed constant Billing targets: 0


### 10. Use solution labels for offline evaluation only

`solution.csv` stores each customer's full Billing configuration as a packed binary string. The next cell checks its alignment with the test IDs and evaluates a simple CRM-configuration lookup baseline. Solution labels are used only to score predictions, never as model inputs or training labels.

In [ ]:
solution = pd.read_csv("solution.csv", dtype={id_col: "string", "Bill_Conf": "string"})
assert solution.columns.tolist() == [id_col, "Bill_Conf"]
assert not solution[id_col].duplicated().any()

solution_aligned = solution.set_index(id_col).reindex(test[id_col].astype("string"))
solution_strings = solution_aligned["Bill_Conf"]
solution_lengths = solution_strings.str.len()
valid_solution = (
    solution_strings.notna().all()
    and solution_lengths.eq(len(bil_cols)).all()
    and solution_strings.str.fullmatch(r"[01]+").all()
)

print("Solution rows:", len(solution))
print("Test rows:", len(test))
print("Solution IDs missing from test:", int((~solution[id_col].isin(test[id_col])).sum()))
print("Test IDs missing from solution:", int(solution_strings.isna().sum()))
print("Solution label lengths:", solution_lengths.value_counts().head().to_dict())

if valid_solution:
    # Pack each binary row into bytes for a compact exact-configuration lookup.
    train_x_packed = np.packbits(df_clean[active_crm].to_numpy(dtype=np.uint8), axis=1)
    train_y_packed = np.packbits(df_clean[active_bil].to_numpy(dtype=np.uint8), axis=1)
    lookup = {
        x.tobytes(): y.tobytes()
        for x, y in zip(train_x_packed, train_y_packed)
    }

    global_y_counts = (
        pd.Series([y.tobytes() for y in train_y_packed])
        .value_counts()
    )
    fallback = global_y_counts.index[0]

    test_x_packed = np.packbits(test[active_crm].to_numpy(dtype=np.uint8), axis=1)
    predicted_packed = [lookup.get(x.tobytes(), fallback) for x in test_x_packed]
    true_packed = [
        np.packbits(np.frombuffer(label.encode("ascii"), dtype=np.uint8) - ord("0")).tobytes()
        for label in solution_strings
    ]

    seen = np.array([x.tobytes() in lookup for x in test_x_packed])
    exact = np.array([pred == truth for pred, truth in zip(predicted_packed, true_packed)])
    bit_errors = np.array([
        int.from_bytes(bytes(a ^ b for a, b in zip(pred, truth)), "big").bit_count()
        for pred, truth in zip(predicted_packed, true_packed)
    ])

    print(f"Exact CRM lookup coverage: {seen.mean():.2%}")
    print(f"Lookup baseline EMR: {exact.mean():.4%}")
    print(f"Lookup baseline mean label error rate: {bit_errors.sum() / (len(test) * len(bil_cols)):.4%}")
    print(f"Seen-configuration EMR: {exact[seen].mean():.4%}" if seen.any() else "No test CRM configurations found in training.")
    print(f"Unseen test configurations: {(~seen).sum():,}")

Solution rows: 97100
Test rows: 97100
Solution IDs missing from test: 0
Test IDs missing from solution: 0
Solution label lengths: {np.int64(731): 97100}
Exact CRM lookup coverage: 0.01%
Lookup baseline EMR: 0.0412%
Lookup baseline mean label error rate: 2.3978%
Seen-configuration EMR: 0.0000%
Unseen test configurations: 97,094


In [ ]:
# Compare two training-only label rules on the same solution benchmark.
# Rule 1: predict each Billing label from its strongest single CRM association.
# Rule 2: predict each Billing label by its training-set majority class.
solution_truth = np.vstack([
    np.frombuffer(label.encode("ascii"), dtype=np.uint8) - ord("0")
    for label in solution_strings
]).astype(np.uint8)

best_crm_row = np.abs(association).argmax(axis=0)
best_phi = association[best_crm_row, np.arange(len(valid_bil))]
test_valid_crm = test[valid_crm].to_numpy(dtype=np.uint8)
selected_values = test_valid_crm[:, best_crm_row]
pair_rule_predictions = np.where(best_phi[None, :] >= 0, selected_values, 1 - selected_values)

majority_values = (df[valid_bil].mean().to_numpy() >= 0.5).astype(np.uint8)
majority_predictions = np.broadcast_to(majority_values, solution_truth.shape)

def report_binary_prediction(name, prediction):
    incorrect = prediction != solution_truth
    row_errors = incorrect.sum(axis=1)
    print(f"{name} EMR: {(row_errors == 0).mean():.4%}")
    print(f"{name} mean label error rate: {incorrect.mean():.4%}")
    print(f"{name} mean incorrect Billing labels per customer: {row_errors.mean():.2f} / {len(valid_bil)}")

print("Best absolute training phi per target:", pd.Series(np.abs(best_phi)).describe().to_dict())
report_binary_prediction("Strongest-pair rule", pair_rule_predictions)
report_binary_prediction("Per-label majority rule", majority_predictions)

pair_target_errors = (pair_rule_predictions != solution_truth).mean(axis=0)
weak_targets = pd.DataFrame({
    "Billing_target": valid_bil,
    "selected_CRM_feature": valid_crm[best_crm_row],
    "training_absolute_phi": np.abs(best_phi),
    "solution_error_rate": pair_target_errors,
}).sort_values("solution_error_rate", ascending=False)
print("Targets with the highest error rate under the strongest-pair rule:")
display(weak_targets.head(15))

Best absolute training phi per target: {'count': 731.0, 'mean': 0.362579882144928, 'std': 0.3545176684856415, 'min': 0.03275461867451668, '25%': 0.04549965634942055, '50%': 0.1811431348323822, '75%': 0.7009299993515015, 'max': 0.9957880973815918}
Strongest-pair rule EMR: 92.9629%
Strongest-pair rule mean label error rate: 0.0137%
Strongest-pair rule mean incorrect Billing labels per customer: 0.10 / 731
Per-label majority rule EMR: 0.0000%
Per-label majority rule mean label error rate: 1.4332%
Per-label majority rule mean incorrect Billing labels per customer: 10.48 / 731
Targets with the highest error rate under the strongest-pair rule:


,Billing_target,selected_CRM_feature,training_absolute_phi,solution_error_rate
57,BIL_3803_PACK,CRM_2106_PACK,0.122878,0.018507
90,BIL_3940_PACK,CRM_1568_PACK,0.303998,0.011432
194,BIL_SUBSCRIBER_STATUS_ORIG_Active,CRM_SUBSCRIBER_STATUS_ORIG_Active,0.988766,0.003491
197,BIL_SUBSCRIBER_STATUS_ORIG_Block_2_Way,CRM_SUBSCRIBER_STATUS_ORIG_Block_2_Way,0.954846,0.003285
98,BIL_3952_PACK,CRM_1598_PACK,0.848281,0.001823
618,BIL_4300_PACK,CRM_1721_PACK,0.035767,0.001473
583,BIL_4349_PACK,CRM_1763_PACK,0.040755,0.001442
702,BIL_4445_PACK,CRM_2177_PACK,0.040027,0.001401
23,BIL_3762_PACK,CRM_1553_PACK,0.992826,0.001236
680,BIL_4421_PACK,CRM_1621_PACK,0.035790,0.001215


### 11. Compare raw and cleaned training strategies

This experiment compares three versions of the same strongest-CRM-pair baseline:

- **Raw:** retain all training rows, including repeated and conflicting examples.
- **Majority:** keep the most frequent complete Billing vector for each CRM configuration.
- **Weighted majority:** keep that winning vector and weight it by its observed frequency.

A grouped holdout keeps each complete CRM configuration entirely in either training or validation. All cleaning and associations are learned from the training fold only. Validation predictions are scored against the original, uncollapsed validation rows using EMR and Hamming loss.

In [8]:
from sklearn.model_selection import GroupShuffleSplit

# Group IDs depend only on CRM inputs, so identical packages stay in one split.
group_codes = df.groupby(crm_cols, sort=False, dropna=False).ngroup().to_numpy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, validation_idx = next(splitter.split(df, groups=group_codes))
fold_train = df.iloc[train_idx][crm_cols + bil_cols]
validation = df.iloc[validation_idx]

# Within the training fold, select the most frequent complete Billing vector per CRM.
fold_combinations = (
    fold_train.groupby(crm_cols + bil_cols, sort=False)
    .size()
    .reset_index(name="_count")
    .sort_values("_count", ascending=False, kind="stable")
)
fold_majority = (
    fold_combinations.drop_duplicates(crm_cols)
    .reset_index(drop=True)
)


def strongest_pair_predict(training_rows, row_weights, validation_rows):
    """Fit strongest single-CRM-feature rules with optional row weights."""
    features = training_rows[crm_cols].to_numpy(dtype=np.uint8)
    targets = training_rows[bil_cols].to_numpy(dtype=np.uint8)
    weights = np.asarray(row_weights, dtype=np.float64)
    total_weight = weights.sum()

    feature_sums = weights @ features
    target_sums = weights @ targets
    cross_products = np.zeros((len(crm_cols), len(bil_cols)), dtype=np.float64)

    # Chunk the matrix product to limit temporary memory use.
    for start in range(0, len(training_rows), 8192):
        stop = start + 8192
        x_chunk = features[start:stop].astype(np.float64)
        y_chunk = targets[start:stop].astype(np.float64)
        weighted_x = x_chunk * weights[start:stop, None]
        cross_products += weighted_x.T @ y_chunk

    numerator = total_weight * cross_products - feature_sums[:, None] * target_sums[None, :]
    feature_denominator = feature_sums * (total_weight - feature_sums)
    target_denominator = target_sums * (total_weight - target_sums)
    denominator = np.sqrt(feature_denominator[:, None] * target_denominator[None, :])
    correlations = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0,
    )
    correlations[feature_denominator == 0, :] = 0

    best_feature = np.abs(correlations).argmax(axis=0)
    best_correlation = correlations[best_feature, np.arange(len(bil_cols))]
    validation_features = validation_rows[crm_cols].to_numpy(dtype=np.uint8)
    predictions = validation_features[:, best_feature].copy()
    predictions[:, best_correlation < 0] = 1 - predictions[:, best_correlation < 0]

    # When no feature has an association, use the (weighted) training target majority.
    no_association = np.abs(best_correlation) <= 1e-12
    target_majority = (target_sums >= total_weight / 2).astype(np.uint8)
    predictions[:, no_association] = target_majority[no_association]
    return predictions


def score_predictions(name, predictions, truth):
    errors = predictions != truth
    exact_match = np.all(~errors, axis=1).mean()
    hamming_loss = errors.mean()
    return {
        "Training strategy": name,
        "Validation rows": len(truth),
        "EMR": exact_match,
        "Hamming loss": hamming_loss,
        "Mean wrong labels per customer": errors.sum(axis=1).mean(),
    }

validation_truth = validation[bil_cols].to_numpy(dtype=np.uint8)
comparison_results = []

# Strategy 1: all raw observations, including conflicting Billing vectors.
raw_predictions = strongest_pair_predict(
    fold_train, np.ones(len(fold_train)), validation
)
comparison_results.append(score_predictions("Raw rows", raw_predictions, validation_truth))

# Strategy 2: one majority Billing vector per CRM configuration, equal group weights.
majority_predictions = strongest_pair_predict(
    fold_majority, np.ones(len(fold_majority)), validation
)
comparison_results.append(score_predictions("Deduplicated majority", majority_predictions, validation_truth))

# Strategy 3: winning vectors weighted by the number of rows supporting each winner.
weighted_predictions = strongest_pair_predict(
    fold_majority, fold_majority["_count"].to_numpy(), validation
)
comparison_results.append(score_predictions("Weighted majority", weighted_predictions, validation_truth))

comparison_table = pd.DataFrame(comparison_results)
print(f"Training rows in fold: {len(train_idx):,}")
print(f"Validation rows scored in original form: {len(validation_idx):,}")
print(f"Unique CRM configurations in training fold: {np.unique(group_codes[train_idx]).size:,}")
display(comparison_table)

C:\Users\raque\AppData\Local\Temp\ipykernel_24388\968198351.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index(name="_count")
C:\Users\raque\AppData\Local\Temp\ipykernel_24388\968198351.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index(name="_count")
C:\Users\raque\AppData\Local\Temp\ipykernel_24388\968198351.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u

Training rows in fold: 151,057
Validation rows scored in original form: 36,385
Unique CRM configurations in training fold: 53,947


,Training strategy,Validation rows,EMR,Hamming loss,Mean wrong labels per customer
0,Raw rows,36385,0.858870,0.002401,1.755091
1,Deduplicated majority,36385,0.847217,0.002442,1.784856
2,Weighted majority,36385,0.869259,0.002382,1.741542


### 12. Train a multi-output decision tree

Run notebook cells from the top through section 11 before running the model cell below. The model uses the `validation` split and scoring function created there; after a kernel restart, those variables must be recreated.

The previous comparison uses CRM/Billing association rules. This section tries a trainable model: one Decision Tree that predicts the 731 Billing outputs together. The tree shares CRM splits across outputs and assigns a binary Billing vector at each leaf. We compare raw rows, one majority Billing vector per CRM configuration, and weighted majority rows on the same CRM-grouped validation split. The weights come only from the training fold.

In [9]:
from scipy.sparse import csr_matrix
from sklearn.tree import DecisionTreeClassifier

X_validation_sparse = csr_matrix(validation[crm_cols].to_numpy(dtype=np.uint8))
y_validation = validation[bil_cols].to_numpy(dtype=np.uint8)


def evaluate_multioutput_tree(name, training_rows, sample_weight=None):
    tree_model = DecisionTreeClassifier(
        max_depth=16,
        min_samples_leaf=50,
        max_features="sqrt",
        random_state=42,
    )
    X_training_sparse = csr_matrix(training_rows[crm_cols].to_numpy(dtype=np.uint8))
    y_training = training_rows[bil_cols].to_numpy(dtype=np.uint8)
    tree_model.fit(X_training_sparse, y_training, sample_weight=sample_weight)
    predictions = tree_model.predict(X_validation_sparse).astype(np.uint8)
    result = score_predictions(name, predictions, y_validation)
    result["Tree leaves"] = tree_model.get_n_leaves()
    return result, tree_model


tree_results = []
raw_tree_result, raw_tree_model = evaluate_multioutput_tree(
    "Raw rows", fold_train
)
tree_results.append(raw_tree_result)

majority_tree_result, majority_tree_model = evaluate_multioutput_tree(
    "Deduplicated majority", fold_majority
)
tree_results.append(majority_tree_result)

weighted_tree_result, weighted_tree_model = evaluate_multioutput_tree(
    "Weighted majority", fold_majority, fold_majority["_count"].to_numpy()
)
tree_results.append(weighted_tree_result)

tree_comparison = pd.DataFrame(tree_results)
print(f"Validation customers: {len(validation):,}")
display(tree_comparison)

Validation customers: 36,385


,Training strategy,Validation rows,EMR,Hamming loss,Mean wrong labels per customer,Tree leaves
0,Raw rows,36385,0.002748,0.007196,5.259942,110
1,Deduplicated majority,36385,0.080198,0.006652,4.862828,93
2,Weighted majority,36385,0.112272,0.006817,4.982960,90


### 13. Compare training strategies across grouped splits

A single split can be lucky or unusually difficult. This section repeats the CRM-grouped holdout with four fixed random seeds and compares raw rows, majority-resolved rows, and frequency-weighted majority rows using the same strongest-feature baseline. Each validation score is calculated on the original, uncollapsed customers. The average EMR selects the training strategy for the next experiments.

In [ ]:
def make_majority_rows(training_rows):
    combinations = (
        training_rows.groupby(crm_cols + bil_cols, sort=False)
        .size()
        .reset_index(name="_count")
        .sort_values("_count", ascending=False, kind="stable")
    )
    return combinations.drop_duplicates(crm_cols).reset_index(drop=True)


def get_strategy_data(training_rows, strategy):
    if strategy == "Raw rows":
        return training_rows, np.ones(len(training_rows), dtype=np.float64)
    majority_rows = make_majority_rows(training_rows)
    if strategy == "Deduplicated majority":
        return majority_rows, np.ones(len(majority_rows), dtype=np.float64)
    return majority_rows, majority_rows["_count"].to_numpy(dtype=np.float64)


strategy_names = ["Raw rows", "Deduplicated majority", "Weighted majority"]
repeated_splitter = GroupShuffleSplit(n_splits=4, test_size=0.2, random_state=2026)
repeated_results = []
per_target_error_sums = {name: np.zeros(len(bil_cols), dtype=np.float64) for name in strategy_names}
per_target_error_counts = {name: np.zeros(len(bil_cols), dtype=np.int64) for name in strategy_names}

for split_number, (fit_idx, valid_idx) in enumerate(
    repeated_splitter.split(df, groups=group_codes), start=1
):
    split_train = df.iloc[fit_idx][crm_cols + bil_cols]
    split_validation = df.iloc[valid_idx]
    split_truth = split_validation[bil_cols].to_numpy(dtype=np.uint8)

    for strategy in strategy_names:
        fit_rows, fit_weights = get_strategy_data(split_train, strategy)
        split_predictions = strongest_pair_predict(fit_rows, fit_weights, split_validation)
        result = score_predictions(strategy, split_predictions, split_truth)
        result["Split"] = split_number
        repeated_results.append(result)

        target_errors = (split_predictions != split_truth).mean(axis=0)
        per_target_error_sums[strategy] += target_errors
        per_target_error_counts[strategy] += 1

repeated_results_table = pd.DataFrame(repeated_results)
strategy_summary = (
    repeated_results_table.groupby("Training strategy")
    .agg(
        mean_EMR=("EMR", "mean"),
        std_EMR=("EMR", "std"),
        mean_Hamming_loss=("Hamming loss", "mean"),
        mean_wrong_labels=("Mean wrong labels per customer", "mean"),
    )
    .sort_values("mean_EMR", ascending=False)
)
selected_strategy = strategy_summary.index[0]
print("Results by split:")
display(repeated_results_table)
print("Average across grouped splits:")
display(strategy_summary)
print("Selected strategy by mean EMR:", selected_strategy)

selected_strategy_target_errors = (
    per_target_error_sums[selected_strategy]
    / per_target_error_counts[selected_strategy]
)
hard_target_indices = np.argsort(selected_strategy_target_errors)[-15:][::-1]
hard_targets = [bil_cols[index] for index in hard_target_indices]
hard_target_report = pd.DataFrame({
    "Billing_target": hard_targets,
    "Grouped_validation_error_rate": selected_strategy_target_errors[hard_target_indices],
})
print("Hardest targets by repeated grouped-validation error rate:")
display(hard_target_report)